In [0]:
-- ============================================================================
-- SQL DATABASE SETUP: TFL Arrivals Endpoint — Raw Layer
-- ============================================================================
-- Purpose: Create a realistic TFL arrivals table with 10,000 messy records
--          for practising SQL clauses, data cleansing, and dbt transformations.
--
-- Source:  Mirrors data from api.tfl.gov.uk/Line/{id}/Arrivals
--          (real-time arrival predictions for tube, bus, DLR, overground)
--
-- Architecture: Azure + Databricks raw schema → dbt staging → dbt mart
--
-- NOTE: No constraints enforced (raw layer accepts everything as-is).
--       Data quality checks belong in the staging/transformation layer.
-- ============================================================================

-- Set catalog & schema context
USE CATALOG `bh2026-winford-uc-dev`;
CREATE SCHEMA IF NOT EXISTS raw;
USE SCHEMA raw;

-- Drop if re-running
DROP TABLE IF EXISTS raw.tfl_arrivals;

-- Create raw arrivals table (mirrors TFL Unified API structure)
CREATE TABLE raw.tfl_arrivals (
    record_id           INT,              -- Surrogate key (should be unique but won't be)
    naptan_id           STRING,           -- Station identifier (e.g., '940GZZLUKSX')
    station_name        STRING,           -- Station display name
    line_id             STRING,           -- Line identifier (e.g., 'northern','victoria')
    line_name           STRING,           -- Line display name
    platform_name       STRING,           -- Platform info (e.g., 'Platform 1')
    direction           STRING,           -- inbound / outbound
    destination_name    STRING,           -- Terminus station
    expected_arrival    TIMESTAMP,        -- Predicted arrival time
    time_to_station     INT,              -- Seconds until arrival at station
    current_location    STRING,           -- Free-text current position
    vehicle_id          STRING,           -- Train/bus vehicle identifier
    mode_name           STRING,           -- tube, bus, dlr, overground, elizabeth-line
    zone                STRING,           -- Fare zone (e.g., '1', '2', '1+2')
    latitude            DECIMAL(9,6),     -- Station latitude
    longitude           DECIMAL(9,6),     -- Station longitude
    ingestion_ts        TIMESTAMP,        -- When this record was ingested
    source_file         STRING            -- Source batch/file identifier
);

In [0]:
-- ============================================================================
-- INSERT 10,000 DELIBERATELY MESSY TFL ARRIVAL RECORDS
-- ============================================================================
-- Data quality issues mapped to each check:
--
-- | Quality Check                        | How It's Represented                          |
-- |--------------------------------------|-----------------------------------------------|
-- | Null/Missing Values                  | NULLs across all columns (~5-15% per field)   |
-- | Primary Key Uniqueness               | Duplicate record_ids (rows 7001-8000 = 1-1000)|
-- | Duplicate Record Detection           | Exact duplicates (rows 9001-9500 = 1-500)     |
-- | Referential Integrity                | line_id/station mismatches (Northern at Pimlico)|
-- | Data Type Validation                 | time_to_station has negative/huge values      |
-- | Numeric Range Validation             | Lat/lon outside London, time > 86400s         |
-- | String Length & Pattern              | Invalid naptan_ids, malformed vehicle_ids     |
-- | Allowed Value / Domain               | Invalid modes ('subway','metro'), directions  |
-- | Business Rule Consistency            | Arrived (time=0) but status 'expected'        |
-- | Cross-Column Consistency             | expected_arrival in past but time_to > 0      |
-- | Timeliness / Freshness               | ingestion_ts from 2019, future timestamps     |
-- | Completeness Check                   | Rows with ALL optional fields NULL            |
-- | Volume Check                         | Duplicates push effective count > 10k         |
-- | Statistical Distribution             | Skewed toward Northern line (~40% in batches) |
-- | Outlier Detection                    | time_to_station = 99999, extreme coordinates  |
-- | Schema Drift Detection               | Mixed casing, extra spaces in categoricals    |
-- | Duplicate File Ingestion             | Same source_file batch repeated (rows 1-500)  |
-- | Negative / Invalid Values            | Negative time_to_station, negative record_id  |
-- | Percentage / Total Consistency       | Lines that don't match station zones          |
-- | Hierarchy Validation                 | Zone values inconsistent with station location|
-- | Audit Column Consistency             | ingestion_ts BEFORE expected_arrival          |
-- ============================================================================

INSERT INTO raw.tfl_arrivals
(record_id, naptan_id, station_name, line_id, line_name, platform_name,
 direction, destination_name, expected_arrival, time_to_station,
 current_location, vehicle_id, mode_name, zone, latitude, longitude,
 ingestion_ts, source_file)

WITH base AS (
    SELECT EXPLODE(SEQUENCE(1, 10000)) AS rn
)
SELECT
    -- RECORD_ID: duplicates (7001-8000 = 1-1000), exact dups (9001-9500 = 1-500), NULLs, negatives
    CASE
        WHEN rn BETWEEN 7001 AND 8000 THEN rn - 7000
        WHEN rn BETWEEN 9001 AND 9500 THEN rn - 9000
        WHEN rn % 500 = 0 THEN NULL
        WHEN rn % 333 = 0 THEN -1 * rn
        ELSE rn
    END AS record_id,

    -- NAPTAN_ID: NULLs, invalid patterns, inconsistent formats, duplicates
    CASE
        WHEN rn % 120 = 0 THEN NULL
        WHEN rn % 67 = 0 THEN 'INVALID'
        WHEN rn % 83 = 0 THEN ''
        WHEN rn % 97 = 0 THEN '940gzzluksx'               -- lowercase (should be upper)
        WHEN rn % 109 = 0 THEN '940GZZLU KSX'             -- space in ID
        WHEN rn % 131 = 0 THEN 'XYZABC123'
        WHEN rn % 11 = 0 THEN '940GZZLUKSX'               -- King's Cross
        WHEN rn % 11 = 1 THEN '940GZZLUVIC'               -- Victoria
        WHEN rn % 11 = 2 THEN '940GZZLUOXC'               -- Oxford Circus
        WHEN rn % 11 = 3 THEN '940GZZLUBND'               -- Bond Street
        WHEN rn % 11 = 4 THEN '940GZZLUWLO'               -- Waterloo
        WHEN rn % 11 = 5 THEN '940GZZLUPCC'               -- Piccadilly Circus
        WHEN rn % 11 = 6 THEN '940GZZLULGN'               -- Paddington
        WHEN rn % 11 = 7 THEN '940GZZLUBKF'               -- Blackfriars
        WHEN rn % 11 = 8 THEN '940GZZLUCPK'               -- Canary Wharf (wrong ID)
        WHEN rn % 11 = 9 THEN '940GZZLUPML'               -- Pimlico
        ELSE '940GZZLUEUS'                                  -- Euston
    END AS naptan_id,

    -- STATION_NAME: NULLs, inconsistent casing, typos, whitespace, mismatched to naptan
    CASE
        WHEN rn % 100 = 0 THEN NULL
        WHEN rn % 73 = 0 THEN ''
        WHEN rn % 47 = 0 THEN "King's Cross St. Pancras"   -- full name
        WHEN rn % 53 = 0 THEN 'Kings Cross'                -- missing apostrophe
        WHEN rn % 59 = 0 THEN 'KINGS CROSS'               -- all caps
        WHEN rn % 61 = 0 THEN ' Victoria '                 -- whitespace
        WHEN rn % 67 = 0 THEN 'Oxfrod Circus'              -- typo
        WHEN rn % 71 = 0 THEN 'WATERLOO'                   -- all caps
        WHEN rn % 79 = 0 THEN 'piccadilly circus'          -- lowercase
        WHEN rn % 83 = 0 THEN 'Paddington  '               -- trailing spaces
        WHEN rn % 89 = 0 THEN 'Balckfriars'                -- typo
        WHEN rn % 11 = 0 THEN "King's Cross St Pancras"
        WHEN rn % 11 = 1 THEN 'Victoria'
        WHEN rn % 11 = 2 THEN 'Oxford Circus'
        WHEN rn % 11 = 3 THEN 'Bond Street'
        WHEN rn % 11 = 4 THEN 'Waterloo'
        WHEN rn % 11 = 5 THEN 'Piccadilly Circus'
        WHEN rn % 11 = 6 THEN 'Paddington'
        WHEN rn % 11 = 7 THEN 'Blackfriars'
        WHEN rn % 11 = 8 THEN 'Canary Wharf'
        WHEN rn % 11 = 9 THEN 'Pimlico'
        ELSE 'Euston'
    END AS station_name,

    -- LINE_ID: NULLs, typos, invalid values, casing issues, wrong lines for stations
    CASE
        WHEN rn % 150 = 0 THEN NULL
        WHEN rn % 43 = 0 THEN 'northen'                    -- typo
        WHEN rn % 57 = 0 THEN 'Northern'                   -- wrong casing
        WHEN rn % 63 = 0 THEN 'VICTORIA'                   -- all caps
        WHEN rn % 77 = 0 THEN 'jubilee'                    -- should match a pattern
        WHEN rn % 87 = 0 THEN 'crossrail'                  -- old name (now elizabeth)
        WHEN rn % 93 = 0 THEN 'metro'                      -- invalid
        WHEN rn % 101 = 0 THEN ''                          -- empty
        WHEN rn % 107 = 0 THEN 'hammersmith-city'          -- hyphenated variant
        WHEN rn % 113 = 0 THEN 'Circle & Hammersmith'      -- wrong format
        WHEN rn % 11 = 9 THEN 'northern'                   -- Pimlico is Victoria line!
        WHEN RAND() < 0.25 THEN 'northern'
        WHEN RAND() < 0.35 THEN 'victoria'
        WHEN RAND() < 0.45 THEN 'central'
        WHEN RAND() < 0.55 THEN 'jubilee'
        WHEN RAND() < 0.65 THEN 'piccadilly'
        WHEN RAND() < 0.75 THEN 'district'
        WHEN RAND() < 0.85 THEN 'circle'
        WHEN RAND() < 0.92 THEN 'elizabeth'
        ELSE 'bakerloo'
    END AS line_id,

    -- LINE_NAME: inconsistent with line_id, NULLs, casing
    CASE
        WHEN rn % 180 = 0 THEN NULL
        WHEN rn % 37 = 0 THEN 'Northern line'              -- extra 'line'
        WHEN rn % 41 = 0 THEN 'VICTORIA LINE'              -- all caps + 'LINE'
        WHEN rn % 51 = 0 THEN 'Central'
        WHEN rn % 61 = 0 THEN 'jubilee'                    -- lowercase
        WHEN rn % 71 = 0 THEN 'Elizabeth Line'
        WHEN rn % 81 = 0 THEN 'Picadilly'                  -- typo
        WHEN RAND() < 0.25 THEN 'Northern'
        WHEN RAND() < 0.4 THEN 'Victoria'
        WHEN RAND() < 0.55 THEN 'Central'
        WHEN RAND() < 0.7 THEN 'Jubilee'
        WHEN RAND() < 0.8 THEN 'Piccadilly'
        WHEN RAND() < 0.9 THEN 'District'
        ELSE 'Circle'
    END AS line_name,

    -- PLATFORM_NAME: NULLs, inconsistent formats, invalid
    CASE
        WHEN rn % 20 = 0 THEN NULL
        WHEN rn % 47 = 0 THEN ''
        WHEN rn % 59 = 0 THEN 'Platform'
        WHEN rn % 67 = 0 THEN 'platform 1'                 -- lowercase
        WHEN rn % 79 = 0 THEN 'PLATFORM 2'                 -- uppercase
        WHEN rn % 89 = 0 THEN 'Plt 3'                      -- abbreviated
        WHEN rn % 97 = 0 THEN 'Platform 99'                -- invalid number
        WHEN rn % 103 = 0 THEN 'Northbound - Platform 1'
        WHEN RAND() < 0.3 THEN 'Platform 1'
        WHEN RAND() < 0.5 THEN 'Platform 2'
        WHEN RAND() < 0.7 THEN 'Platform 3'
        ELSE 'Platform 4'
    END AS platform_name,

    -- DIRECTION: NULLs, invalid values, casing, old formats
    CASE
        WHEN rn % 200 = 0 THEN NULL
        WHEN rn % 43 = 0 THEN 'Inbound'                    -- wrong casing
        WHEN rn % 53 = 0 THEN 'OUTBOUND'                   -- all caps
        WHEN rn % 61 = 0 THEN 'North'                      -- invalid value
        WHEN rn % 71 = 0 THEN 'Southbound'                 -- old format
        WHEN rn % 79 = 0 THEN 'eastbound'                  -- old format
        WHEN rn % 89 = 0 THEN ''
        WHEN rn % 97 = 0 THEN 'both'                       -- invalid
        WHEN RAND() < 0.5 THEN 'inbound'
        ELSE 'outbound'
    END AS direction,

    -- DESTINATION_NAME: NULLs, inconsistent, typos, impossible destinations
    CASE
        WHEN rn % 90 = 0 THEN NULL
        WHEN rn % 37 = 0 THEN ''
        WHEN rn % 47 = 0 THEN 'High Barent'                -- typo (High Barnet)
        WHEN rn % 57 = 0 THEN 'MORDEN'                     -- all caps
        WHEN rn % 67 = 0 THEN 'edgware'                    -- lowercase
        WHEN rn % 77 = 0 THEN 'Heathrow Terminal 5'
        WHEN rn % 87 = 0 THEN 'Stratford International'    -- wrong network (HS1)
        WHEN rn % 97 = 0 THEN 'Check Front of Train'
        WHEN rn % 107 = 0 THEN ' Brixton '                 -- whitespace
        WHEN RAND() < 0.15 THEN 'High Barnet'
        WHEN RAND() < 0.25 THEN 'Morden'
        WHEN RAND() < 0.35 THEN 'Edgware'
        WHEN RAND() < 0.45 THEN 'Brixton'
        WHEN RAND() < 0.55 THEN 'Epping'
        WHEN RAND() < 0.65 THEN 'Heathrow Terminal 4'
        WHEN RAND() < 0.75 THEN 'Stanmore'
        WHEN RAND() < 0.85 THEN 'Stratford'
        ELSE 'Cockfosters'
    END AS destination_name,

    -- EXPECTED_ARRIVAL: NULLs, future dates, past dates, epoch, far future
    CASE
        WHEN rn % 80 = 0 THEN NULL
        WHEN rn % 111 = 0 THEN CAST('2099-12-31 23:59:59' AS TIMESTAMP)
        WHEN rn % 131 = 0 THEN CAST('1970-01-01 00:00:00' AS TIMESTAMP)
        WHEN rn % 151 = 0 THEN CAST('2019-03-15 08:30:00' AS TIMESTAMP)   -- very old
        WHEN rn % 171 = 0 THEN CAST(DATE_ADD(CURRENT_DATE, 365) AS TIMESTAMP)
        ELSE CAST(CONCAT(CURRENT_DATE, ' ', LPAD(CAST(CAST(RAND() * 23 AS INT) AS STRING), 2, '0'), ':', LPAD(CAST(CAST(RAND() * 59 AS INT) AS STRING), 2, '0'), ':00') AS TIMESTAMP)
    END AS expected_arrival,

    -- TIME_TO_STATION: NULLs, negatives, outliers, impossibly large
    CASE
        WHEN rn % 25 = 0 THEN NULL
        WHEN rn % 113 = 0 THEN -120                        -- negative (arrived already?)
        WHEN rn % 127 = 0 THEN 99999                       -- impossible (27+ hours)
        WHEN rn % 137 = 0 THEN 0                           -- already there
        WHEN rn % 149 = 0 THEN 86400                       -- exactly 24 hours
        WHEN rn % 163 = 0 THEN -1
        WHEN rn BETWEEN 8001 AND 8200 THEN CAST(RAND() * 7200 + 3600 AS INT)  -- 1-3 hours (suspicious)
        ELSE CAST(RAND() * 1800 AS INT)                    -- normal: 0-30 mins
    END AS time_to_station,

    -- CURRENT_LOCATION: NULLs, inconsistent, free-text mess
    CASE
        WHEN rn % 10 = 0 THEN NULL
        WHEN rn % 37 = 0 THEN ''
        WHEN rn % 47 = 0 THEN 'Between Stations'
        WHEN rn % 57 = 0 THEN 'between stations'           -- lowercase
        WHEN rn % 67 = 0 THEN 'At Platform'
        WHEN rn % 77 = 0 THEN 'AT PLATFORM'                -- all caps
        WHEN rn % 87 = 0 THEN 'Approaching Station'
        WHEN rn % 97 = 0 THEN 'Left Previous Station'
        WHEN rn % 107 = 0 THEN 'Unknown'
        WHEN rn % 117 = 0 THEN 'N/A'
        WHEN RAND() < 0.3 THEN 'Between Euston and Kings Cross'
        WHEN RAND() < 0.5 THEN 'Approaching Oxford Circus'
        WHEN RAND() < 0.7 THEN 'At Victoria'
        ELSE 'Left Waterloo'
    END AS current_location,

    -- VEHICLE_ID: NULLs, invalid formats, duplicates, inconsistent
    CASE
        WHEN rn % 15 = 0 THEN NULL
        WHEN rn % 73 = 0 THEN ''
        WHEN rn % 83 = 0 THEN 'UNKNOWN'
        WHEN rn % 97 = 0 THEN 'TBC'
        WHEN rn % 109 = 0 THEN 'train-001'                 -- wrong format
        WHEN rn % 127 = 0 THEN '000'                       -- suspiciously short
        WHEN rn BETWEEN 9001 AND 9500 THEN CONCAT('V', LPAD(CAST(rn - 9000 AS STRING), 4, '0'))
        ELSE CONCAT('V', LPAD(CAST(CAST(RAND() * 500 + 1 AS INT) AS STRING), 4, '0'))
    END AS vehicle_id,

    -- MODE_NAME: NULLs, invalid values, casing, old/wrong terminology
    CASE
        WHEN rn % 250 = 0 THEN NULL
        WHEN rn % 43 = 0 THEN 'Tube'                       -- wrong casing
        WHEN rn % 53 = 0 THEN 'TUBE'                       -- all caps
        WHEN rn % 61 = 0 THEN 'subway'                     -- US terminology
        WHEN rn % 71 = 0 THEN 'metro'                      -- European
        WHEN rn % 79 = 0 THEN 'underground'                -- colloquial
        WHEN rn % 89 = 0 THEN 'Bus'
        WHEN rn % 97 = 0 THEN ''
        WHEN rn % 107 = 0 THEN 'overground'                -- valid but inconsistent batch
        WHEN rn % 113 = 0 THEN 'DLR'
        WHEN rn % 127 = 0 THEN 'crossrail'                 -- old name
        ELSE 'tube'
    END AS mode_name,

    -- ZONE: NULLs, inconsistent formats, invalid, doesn't match station
    CASE
        WHEN rn % 60 = 0 THEN NULL
        WHEN rn % 43 = 0 THEN 'Zone 1'                     -- should just be '1'
        WHEN rn % 53 = 0 THEN 'zone 2'                     -- lowercase prefix
        WHEN rn % 61 = 0 THEN '1+2'                        -- valid but different format
        WHEN rn % 71 = 0 THEN '0'                          -- invalid zone
        WHEN rn % 79 = 0 THEN '99'                         -- impossible zone
        WHEN rn % 89 = 0 THEN 'N/A'
        WHEN rn % 97 = 0 THEN ''
        WHEN rn % 103 = 0 THEN '1/2'                       -- wrong separator
        WHEN rn % 11 = 9 THEN '3'                          -- Pimlico is Zone 1!
        WHEN RAND() < 0.4 THEN '1'
        WHEN RAND() < 0.6 THEN '2'
        WHEN RAND() < 0.8 THEN '3'
        ELSE '4'
    END AS zone,

    -- LATITUDE: NULLs, out of London range, swapped with longitude, zeros
    CASE
        WHEN rn % 70 = 0 THEN NULL
        WHEN rn % 113 = 0 THEN 0.000000                    -- null island
        WHEN rn % 127 = 0 THEN -0.118092                   -- swapped with longitude!
        WHEN rn % 137 = 0 THEN 90.000000                   -- North Pole
        WHEN rn % 149 = 0 THEN 48.856614                   -- Paris latitude
        WHEN rn % 163 = 0 THEN 51.000000                   -- too far south of London
        WHEN rn % 11 = 0 THEN 51.530663                    -- King's Cross
        WHEN rn % 11 = 1 THEN 51.496394                    -- Victoria
        WHEN rn % 11 = 2 THEN 51.515224                    -- Oxford Circus
        WHEN rn % 11 = 3 THEN 51.514304                    -- Bond Street
        WHEN rn % 11 = 4 THEN 51.503299                    -- Waterloo
        WHEN rn % 11 = 5 THEN 51.509974                    -- Piccadilly Circus
        WHEN rn % 11 = 6 THEN 51.516021                    -- Paddington
        WHEN rn % 11 = 7 THEN 51.511934                    -- Blackfriars
        WHEN rn % 11 = 8 THEN 51.503538                    -- Canary Wharf
        WHEN rn % 11 = 9 THEN 51.489468                    -- Pimlico
        ELSE 51.527864                                      -- Euston
    END AS latitude,

    -- LONGITUDE: NULLs, swapped with lat, zeros, out of range
    CASE
        WHEN rn % 70 = 0 THEN NULL
        WHEN rn % 113 = 0 THEN 0.000000                    -- null island
        WHEN rn % 127 = 0 THEN 51.530663                   -- swapped with latitude!
        WHEN rn % 137 = 0 THEN 0.000000                    -- on prime meridian but wrong
        WHEN rn % 149 = 0 THEN 2.352222                    -- Paris longitude
        WHEN rn % 163 = 0 THEN -5.000000                   -- too far west
        WHEN rn % 11 = 0 THEN -0.123924                    -- King's Cross
        WHEN rn % 11 = 1 THEN -0.144544                    -- Victoria
        WHEN rn % 11 = 2 THEN -0.141903                    -- Oxford Circus
        WHEN rn % 11 = 3 THEN -0.149536                    -- Bond Street
        WHEN rn % 11 = 4 THEN -0.113387                    -- Waterloo
        WHEN rn % 11 = 5 THEN -0.134454                    -- Piccadilly Circus
        WHEN rn % 11 = 6 THEN -0.175783                    -- Paddington
        WHEN rn % 11 = 7 THEN -0.103782                    -- Blackfriars
        WHEN rn % 11 = 8 THEN -0.019440                    -- Canary Wharf
        WHEN rn % 11 = 9 THEN -0.133528                    -- Pimlico
        ELSE -0.133009                                      -- Euston
    END AS longitude,

    -- INGESTION_TS: NULLs, far future, far past, before expected_arrival
    CASE
        WHEN rn % 30 = 0 THEN NULL
        WHEN rn % 121 = 0 THEN CAST('2099-01-01 00:00:00' AS TIMESTAMP)   -- far future
        WHEN rn % 139 = 0 THEN CAST('2019-06-01 12:00:00' AS TIMESTAMP)   -- years old
        WHEN rn % 157 = 0 THEN CAST('1970-01-01 00:00:00' AS TIMESTAMP)   -- epoch
        WHEN rn % 179 = 0 THEN CAST(DATE_SUB(CURRENT_DATE, 730) AS TIMESTAMP)  -- 2 years ago
        ELSE CURRENT_TIMESTAMP
    END AS ingestion_ts,

    -- SOURCE_FILE: NULLs, duplicated batches, inconsistent naming
    CASE
        WHEN rn % 250 = 0 THEN NULL
        WHEN rn % 119 = 0 THEN ''
        WHEN rn BETWEEN 9001 AND 9500 THEN 'tfl_arrivals_batch_001.json'   -- DUPLICATE batch!
        WHEN rn BETWEEN 1 AND 500 THEN 'tfl_arrivals_batch_001.json'       -- same file name
        WHEN rn BETWEEN 501 AND 1500 THEN 'tfl_arrivals_batch_002.json'
        WHEN rn BETWEEN 1501 AND 2500 THEN 'tfl_arrivals_batch_003.json'
        WHEN rn BETWEEN 2501 AND 3500 THEN 'TFL_Arrivals_Batch_004.JSON'   -- inconsistent naming!
        WHEN rn BETWEEN 3501 AND 4500 THEN 'tfl_arrivals_batch_005.json'
        WHEN rn BETWEEN 4501 AND 5500 THEN 'tfl_arrivals_batch_006.json'
        WHEN rn BETWEEN 5501 AND 6500 THEN 'tfl_arrivals_BATCH_007.json'   -- inconsistent
        WHEN rn BETWEEN 6501 AND 7000 THEN 'tfl_arrivals_batch_008.json'
        WHEN rn BETWEEN 7001 AND 8000 THEN 'tfl_arrivals_batch_009_RERUN.json'  -- rerun file
        WHEN rn BETWEEN 8001 AND 9000 THEN 'tfl_arrivals_batch_010.json'
        ELSE 'tfl_arrivals_batch_011.json'
    END AS source_file

FROM base;